# 19 Nested feature selection check

The reduced feature set R ("core", 10 features) of notebook 09 beat the full set by about 0.33-0.46 MAE, but it was chosen by looking at permutation importances on validation folds 8 and 16, so that edge is optimistic. The current candidate uses all V0 inputs (no selection). This notebook asks whether selecting features *using only the training data of each fold* helps, and how much of R's apparent edge survives an honest selection.

GBM only (the structure of D-008, split cutoffs: 09:00 with demand, 07:00 without), `forecast = ref28 + L(D) + centred GBM deviation`; the level `L(D)` is identical in all variants, so the differences are due to the within-day shape model. Variants, fixed before running:

| Variant | Features |
| --- | --- |
| full | all V0 inputs (as in notebook 18) |
| R core | the 10 features of notebook 09 (chosen on validation folds; 9 at 07:00 without `demand_mw`) |
| N1 | nested: keep features with positive permutation importance |
| N2 | nested: keep the 10 features with the highest permutation importance |

Nested selection, inside each fold using the training days only: fit the GBM on the training days except the last 56 (and one gap day); compute the permutation importance of each feature on those last 56 days (increase in MAE of the predicted deviation when the feature is shuffled; 3 shuffles, fixed random state); select; refit on all training days with the selected features. Same 17 folds, dev-only.

## Init

In [1]:
import sys, time
sys.path[:0] = ["notebooks", "."]  # _common imports from repo root or notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from _eval import make_folds, score

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100})

TT = pd.read_parquet("outputs/feature_table_dev.parquet").reset_index(drop=True)
REF = "med28_same"
days = pd.DatetimeIndex(sorted(TT["day"].unique()))
folds = make_folds(days, 28, 17, 1)
val_days_all = pd.DatetimeIndex(np.concatenate([f["val_days"].values for f in folds]))
COMMON = TT[REF].notna() & TT["price"].notna()
TT["ydev"] = TT["price"] - TT[REF]
CUT = {7: TT["cutoff_hour"] == 7, 9: TT["cutoff_hour"] == 9}

ABS = ["demand_mw", "wind_mw", "emb_wind_mw", "emb_solar_mw"]
ANOM = ["demand_anom", "wind_anom", "emb_wind_anom", "emb_solar_anom"]
HIST = ["med28_same", "med7_same", "d2_same", "level24h", "level7", "level28", "level_gap"]
MGN = ["mgn_min_demand", "mgn_surplus", "mgn_constrained", "mgn_gen_avail", "mgn_ic_import"]
GBM_EXTRA = ["wind_total_mw", "demand_age_h", "wind_age_h", "emb_age_h", "demand_rev", "wind_rev", "emb_wind_rev", "emb_solar_rev", "is_weekend"]
DEMAND_COLS = {"demand_mw", "demand_age_h", "demand_rev", "demand_anom"}
REV_COLS = {"demand_rev", "wind_rev", "emb_wind_rev", "emb_solar_rev"}
DROP = {9: set(), 7: DEMAND_COLS | REV_COLS}
GCOLS = {c: [x for x in dict.fromkeys(["hour_frac", "weekday", "bank_holiday", "doy_sin", "doy_cos", "clock_change_day"] + ABS + MGN + GBM_EXTRA + HIST) if x not in DROP[c]] for c in (7, 9)}
LCOLS = {9: ANOM + ["is_weekend", "bank_holiday"], 7: [a for a in ANOM if a != "demand_anom"] + ["is_weekend", "bank_holiday"]}

dm = TT[COMMON & CUT[9]].groupby("day")["ydev"].agg(["mean", "count"])
DAILY = {}
for c in (7, 9):
    g = TT[CUT[c]].groupby("day")
    d = g[ANOM].mean(); d["is_weekend"] = g["is_weekend"].mean(); d["bank_holiday"] = g["bank_holiday"].mean()
    d["target"] = dm["mean"].where(dm["count"] >= 40).reindex(d.index)
    DAILY[c] = d

def gbm(seed):
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                         min_samples_leaf=100, l2_regularization=1.0, early_stopping=False, random_state=seed)
def level_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), HuberRegressor(alpha=10, epsilon=1.35, max_iter=300))

def fold_table(pred, cut):
    rows = []
    for f in folds:
        d = TT[CUT[cut] & COMMON & TT["day"].isin(f["val_days"])].assign(pred=pred)
        s = score(d, "pred"); s["fold"] = f["fold"]; rows.append(s)
    return pd.DataFrame(rows).set_index("fold")
BASE = {c: fold_table(TT[REF], c) for c in (7, 9)}
print("ready; baseline MAE", {c: round(BASE[c]["MAE"].mean(), 3) for c in BASE})

ready; baseline MAE {7: np.float64(30.203), 9: np.float64(30.203)}


## 1. Variants and nested selection

In [2]:
CORE = ["hour_frac", "weekday", "is_weekend", "bank_holiday", "med7_same", "med28_same", "demand_mw", "wind_mw", "emb_wind_mw", "emb_solar_mw"]
CORE_COLS = {c: [x for x in CORE if x not in DROP[c]] for c in (7, 9)}
print({c: len(GCOLS[c]) for c in GCOLS}, "full features;", {c: len(CORE_COLS[c]) for c in CORE_COLS}, "core features")

def perm_importance(cut, train_days, cols, n_rep=3, inner=56):
    td = pd.DatetimeIndex(sorted(train_days))
    va_d, tr_d = td[-inner:], td[:-(inner + 1)]
    tr = TT[CUT[cut] & TT["day"].isin(tr_d) & COMMON]; va = TT[CUT[cut] & TT["day"].isin(va_d) & COMMON]
    m = gbm(0).fit(tr[cols], tr["ydev"])
    Xv = va[cols].copy(); y = va["ydev"].to_numpy()
    base = np.abs(y - m.predict(Xv)).mean()
    rng = np.random.default_rng(0); imp = {}
    for c in cols:
        vals = []
        for _ in range(n_rep):
            Xp = Xv.copy(); Xp[c] = rng.permutation(Xp[c].to_numpy())
            vals.append(np.abs(y - m.predict(Xp)).mean() - base)
        imp[c] = float(np.mean(vals))
    return pd.Series(imp)

def run_variants(cut):
    out = {v: pd.Series(np.nan, index=TT.index) for v in ("full", "R core", "N1", "N2")}
    lvl = pd.Series(np.nan, index=DAILY[cut].index)
    sel = {"N1": [], "N2": []}
    full = GCOLS[cut]
    for f in folds:
        tr = TT[CUT[cut] & TT["day"].isin(f["train_days"]) & COMMON]
        va = TT[CUT[cut] & TT["day"].isin(f["val_days"])]
        imp = perm_importance(cut, f["train_days"], full)
        n1 = [c for c in full if imp[c] > 0] or [imp.idxmax()]
        n2 = list(imp.sort_values(ascending=False).index[:10])
        sel["N1"].append(n1); sel["N2"].append(n2)
        for v, cols in (("full", full), ("R core", CORE_COLS[cut]), ("N1", n1), ("N2", n2)):
            out[v].loc[va.index] = gbm(0).fit(tr[cols], tr["ydev"]).predict(va[cols])
        dtr = DAILY[cut].loc[DAILY[cut].index.isin(f["train_days"]) & DAILY[cut]["target"].notna()]
        lm = level_model().fit(dtr[LCOLS[cut]], dtr["target"])
        dva = DAILY[cut].loc[DAILY[cut].index.isin(f["val_days"])]
        lvl.loc[dva.index] = lm.predict(dva[LCOLS[cut]])
    res = {}
    for v, d0 in out.items():
        d0c = d0 - d0.groupby(TT["day"]).transform("mean")
        res[v] = (TT[REF] + TT["day"].map(lvl) + d0c).where(CUT[cut])
    return res, sel

PR, SEL = {}, {}
t0 = time.time()
for cut in (9, 7):
    r, s = run_variants(cut)
    for v, p in r.items():
        PR[(cut, v)] = p
    SEL[cut] = s
    print("cutoff", cut, f"{time.time() - t0:.0f}s")
P18 = pd.read_parquet("outputs/cv_preds_families_0700_0900.parquet")
PT = TT[["period_start", "cutoff_hour"]].merge(P18, on=["period_start", "cutoff_hour"], how="left"); PT.index = TT.index
for cut in (9, 7):
    a = PR[(cut, "full")]; b = PT[f"{cut} | GBM seed 0"]; ok = a.notna() & b.notna()
    print(cut, "full vs notebook 18 GBM, max abs difference:", float((a[ok] - b[ok]).abs().max()))

{7: 25, 9: 31} full features; {7: 9, 9: 10} core features


cutoff 9 103s


cutoff 7 198s
9 full vs notebook 18 GBM, max abs difference: 0.0
7 full vs notebook 18 GBM, max abs difference: 0.0


## 2. Scores

Differences to the 28-day baseline at the same cutoff; `n feat` is the mean number of features used (nested variants vary by fold).

In [3]:
def row(cut, pred):
    ft = fold_table(pred, cut); b = BASE[cut]
    d = {m: ft[m] - b[m] for m in ["MAE", "RMSE", "hit_top8", "hit_bot8", "MAE_top10", "MAE_bot10"]}
    return ft, {"dMAE all": d["MAE"].mean(), "dMAE 0-12": d["MAE"].loc[:12].mean(), "dMAE late": d["MAE"].loc[13:].mean(), "dRMSE all": d["RMSE"].mean(),
                "dHitTop": d["hit_top8"].mean(), "dHitBot": d["hit_bot8"].mean(), "dTop10": d["MAE_top10"].mean(), "dBot10": d["MAE_bot10"].mean(),
                "bias late": ft["bias"].loc[13:].mean()}
FT, rows = {}, {}
for (cut, v), p in PR.items():
    ft, r = row(cut, p); FT[(cut, v)] = ft
    r["n feat"] = {"full": len(GCOLS[cut]), "R core": len(CORE_COLS[cut]), "N1": np.mean([len(x) for x in SEL[cut]["N1"]]), "N2": 10}[v]
    rows[f"{cut:02d}:00 {v}"] = r
print(pd.DataFrame(rows).T.round(3).to_string())

              dMAE all  dMAE 0-12  dMAE late  dRMSE all  dHitTop  dHitBot  dTop10  dBot10  bias late  n feat
09:00 full      -2.043     -2.176     -1.612     -4.064    0.023    0.040  -4.964 -23.258     -2.185  31.000
09:00 R core    -2.024     -2.020     -2.038     -3.841    0.017    0.032  -5.614 -24.983     -2.185  10.000
09:00 N1        -1.949     -2.127     -1.370     -4.009    0.021    0.029  -5.005 -22.602     -2.185  16.765
09:00 N2        -1.936     -2.080     -1.469     -3.955    0.023    0.029  -4.893 -22.905     -2.185  10.000
07:00 full      -1.696     -1.888     -1.073     -3.552    0.018    0.028  -3.772 -20.689     -1.584  25.000
07:00 R core    -1.573     -1.496     -1.823     -3.005    0.011    0.031  -4.192 -22.295     -1.584   9.000
07:00 N1        -1.780     -1.940     -1.259     -3.602    0.014    0.028  -3.553 -21.090     -1.584  14.588
07:00 N2        -1.834     -1.988     -1.334     -3.629    0.011    0.027  -3.739 -21.794     -1.584  10.000


## 3. Paired comparisons

First minus second; mean per-fold difference +/- standard error over 17 folds (negative better for MAE/RMSE, positive for hit rates); `late` is folds 13-16; last column the number of folds where the first has the lower MAE.

In [4]:
def paired(a, b):
    r = {}
    for m in ["MAE", "RMSE", "hit_top8", "hit_bot8"]:
        d = FT[a][m] - FT[b][m]
        r[f"{m} all"] = f"{d.mean():+.3f} +/- {d.std() / np.sqrt(17):.3f}"; r[f"{m} late"] = f"{d.loc[13:].mean():+.3f}"
    r["MAE better in"] = int(((FT[a]["MAE"] - FT[b]["MAE"]) < 0).sum())
    return r
rows = {}
for cut in (9, 7):
    for a, b in [("R core", "full"), ("N1", "full"), ("N2", "full"), ("R core", "N2")]:
        rows[f"{cut:02d}:00 {a} - {b}"] = paired((cut, a), (cut, b))
print(pd.DataFrame(rows).T.to_string())

                              MAE all MAE late          RMSE all RMSE late      hit_top8 all hit_top8 late      hit_bot8 all hit_bot8 late MAE better in
09:00 R core - full  +0.019 +/- 0.135   -0.426  +0.223 +/- 0.159    -0.268  -0.006 +/- 0.006        -0.012  -0.008 +/- 0.007        -0.017            10
09:00 N1 - full      +0.095 +/- 0.086   +0.241  +0.054 +/- 0.101    +0.187  -0.002 +/- 0.004        -0.008  -0.011 +/- 0.006        -0.035             9
09:00 N2 - full      +0.107 +/- 0.124   +0.143  +0.109 +/- 0.137    +0.137  -0.000 +/- 0.005        -0.013  -0.011 +/- 0.005        -0.009             8
09:00 R core - N2    -0.088 +/- 0.117   -0.569  +0.114 +/- 0.147    -0.404  -0.005 +/- 0.007        +0.001  +0.003 +/- 0.006        -0.008            11
07:00 R core - full  +0.123 +/- 0.200   -0.750  +0.547 +/- 0.288    -0.730  -0.007 +/- 0.007        +0.017  +0.003 +/- 0.008        +0.019             7
07:00 N1 - full      -0.083 +/- 0.082   -0.186  -0.050 +/- 0.105    -0.163  -0.003

## 4. Which features are selected

Number of folds (of 17) in which each feature was kept by the nested rules, for N1 and N2.

In [5]:
for cut in (9, 7):
    cnt = pd.DataFrame({v: pd.Series([c for fl in SEL[cut][v] for c in fl]).value_counts() for v in ("N1", "N2")}).reindex(GCOLS[cut]).fillna(0).astype(int)
    cnt["in R core"] = [c in CORE_COLS[cut] for c in cnt.index]
    print(f"{cut:02d}:00"); print(cnt.sort_values("N2", ascending=False).to_string()); print()
out = pd.DataFrame({f"{c} | {v}": p for (c, v), p in PR.items()})
for c in ["day", "period_start", "cutoff_hour", "price", REF]:
    out[c] = TT[c]
out[out["day"].isin(val_days_all)].to_parquet("outputs/cv_preds_selection_0700_0900.parquet", index=False)

09:00
                  N1  N2  in R core
demand_mw         17  17       True
med28_same        17  17       True
emb_solar_mw      15  15       True
emb_wind_mw       15  15       True
mgn_surplus       13  10      False
level28           12  10      False
mgn_min_demand    11  10      False
mgn_gen_avail     13   9      False
wind_total_mw     10   8      False
wind_mw           10   8       True
hour_frac         10   7       True
emb_solar_rev     11   5      False
emb_wind_rev      12   5      False
doy_cos            8   4      False
doy_sin            6   4      False
emb_age_h         10   4      False
level_gap          7   4      False
wind_age_h         9   4      False
level7            10   3      False
mgn_constrained   11   3      False
med7_same          5   2       True
level24h           6   2      False
weekday           12   2       True
mgn_ic_import      8   1      False
demand_age_h       7   1      False
bank_holiday       4   0       True
clock_change_day   0  

## Findings (GBM within the D-008 structure, 17 folds; variants fixed before running; differences to the 28-day baseline at the same cutoff)

- **Selecting features inside each fold does not help.** Against using all inputs: at 09:00 N1 (positive permutation importance, on average 17 of 31 features) is +0.10 +/- 0.09 MAE (better in 9 of 17 folds) and N2 (top 10) +0.11 +/- 0.12 (8 of 17); at 07:00 N1 is -0.08 +/- 0.08 (9 of 17) and N2 -0.14 +/- 0.10 (12 of 17). RMSE and hit rates are within about two standard errors of zero, the hit rates slightly lower with selection (bottom-8 -0.011 at 09:00 for both rules, top-8 -0.007 for N2 at 07:00). So none of the rules is distinguishable from the full set, and the full set is the simplest.
- **The apparent edge of R ("core") does not carry over to this structure.** In notebook 09 (direct price, no level part) R beat the full set by about 0.33-0.46 MAE. Here, with the same structure for all variants, R is +0.02 +/- 0.14 MAE at 09:00 and +0.12 +/- 0.20 at 07:00 relative to the full set, with worse RMSE (+0.22 +/- 0.16 and +0.55 +/- 0.29) and a slightly lower top-8 hit rate (-0.006 at 09:00, -0.007 at 07:00, about one standard error; at 07:00 hit gain +0.011 against +0.018 with all features). Part of the earlier advantage was therefore either the optimistic selection or because the level part (notebook 15) now removes what R was compensating for. In the late folds R is better on MAE (-0.43 at 09:00, -0.75 at 07:00), and late-fold RMSE is also better (-0.27 / -0.73), but this is four folds and the all-fold RMSE and hit rates are worse; it is not enough to act on.
- **Which features are chosen is fairly stable at the top and noisy below.** At 09:00 `demand_mw` and `med28_same` are kept in all 17 folds by both rules, `emb_wind_mw` and `emb_solar_mw` in 15, `wind_mw` in 8-10; at 07:00 `emb_solar_mw` and `med28_same` in all 17, `hour_frac`, `mgn_min_demand`, `emb_wind_mw` in 15-16, `wind_mw` in 13-14. `is_weekend` and `clock_change_day` are never selected at 09:00 and `is_weekend` never at 07:00. Margin columns (`mgn_surplus`, `mgn_min_demand`, `mgn_gen_avail`) are selected in a majority of folds, more often than I expected from notebook 13. Many mid-ranked features switch in and out from fold to fold (selection counts 2-12 of 17), as expected when features are correlated and permutation importance splits credit between them.
- **Decision for the candidate.** Keep all V0 inputs, no feature selection (simplest; the selection step does not add measurable value in this structure).
- **Caveats.** (1) Permutation importance on a single inner window (the last 56 training days, 3 shuffles) is a noisy and, for correlated features, a biased guide; other selection rules (stability selection, a different window, grouped features) are untried. (2) GBM only: the Huber shape model has its own (spline) feature design and was not tested. (3) The level model's inputs were not selected. (4) Differences are about 1 standard error or less in all-fold MAE, so the conclusion is "no evidence of a benefit", not "evidence of no benefit". (5) The structure itself was designed on these folds (D-008 caveat).
